# Subalgebras, ideals and quotients

This notebook shows the constructions that live inside a vertex algebra $V$: the subalgebra generated by some elements, the ideal generated by some elements, and the quotient by an ideal. It explains which answers are exact and which hold only up to a weight. It then builds the Virasoro minimal models as quotients by singular vectors and compares them with their vacuum characters, and it ends with presentations of a subalgebra by strong generators, OPEs and relations.

**Prerequisites:** `01_getting_started.ipynb`.

**Contents**

1. Subalgebras
2. Exact answers and answers up to a weight
3. Ideals
4. Quotients
5. Minimal models
6. Presentations

## 1. Subalgebras

`V.subalgebra(A)` is the vertex subalgebra $S$ generated by the homogeneous elements $A$. The library finds a set $G$ of strong generators of $S$: every element of $S$ is a linear combination of normally ordered monomials ${:}\partial^{k_1}g_1 \cdots \partial^{k_r}g_r{:}$ with $g_i \in G$. It starts with $G = A$ and tests the products $g_{(n)}g'$, $n \ge 0$. A product that is not a combination of monomials in $G$ becomes a new strong generator, and the test is repeated with it. When a round adds nothing, $G$ strongly generates $S$.

The first example is the Heisenberg algebra $J(z)J(w) \sim 1/(z-w)^2$ with conformal vector $T = \frac12{:}JJ{:}$ (central charge 1), and the subalgebra generated by $T$. `S.strong_generator_info()` lists each strong generator with its conformal weight, parity and origin.

In [1]:
import vertex_algebras as va
from vertex_algebras import *

X.<J> = NormallyOrderedExpressions(QQ)
H = UniversalVertexAlgebra(QQ, X, {('J', 'J'): [0, 1]}, conformal_vector=J*J/2)
J = H.gen('J')
T = H.conformal_vector()
S = H.subalgebra([T], names=['T'])
S, S.strong_generator_info()

(Vertex subalgebra of Universal vertex algebra on generators ['J'] over Rational Field strongly generated by ['T'],
 [('T', 2, 0, 'given')])

The closure stops at once: $T$ alone strongly generates $S$. The weight spaces of $S$ are smaller than those of $H$. Their dimensions agree with the numbers of partitions of $n$ into parts $\ge 2$, the dimensions of the universal Virasoro vertex algebra. So up to weight 13 the map from the universal Virasoro algebra at $c = 1$ onto $S$ is an isomorphism; the code checks nothing above weight 13.

In [2]:
[H.weight_space(n).dimension() for n in range(14)], [S.weight_space(n).dimension() for n in range(14)]

([1, 1, 2, 3, 5, 7, 11, 15, 22, 30, 42, 56, 77, 101],
 [1, 0, 1, 1, 2, 2, 4, 4, 7, 8, 12, 14, 21, 24])

In [3]:
[S.weight_space(n).dimension() for n in range(14)] == [Partitions(n, min_part=2).cardinality() for n in range(14)]

True

**Membership.** `S.contains(v)`, or `v in S`, decides exactly whether an element $v$ of $H$ lies in $S$. Here ${:}J\,\partial J{:} = \partial T$ lies in $S$, while $J$ and ${:}JJJ{:}$ do not.

In [4]:
J in S, T.no(T) in S, J.no(J.derivative()) in S, J.no(J.no(J)) in S

(False, True, True, False)

**Expressions in the strong generators.** `S.express(v)` writes an element of $S$ as a combination of normally ordered monomials of the strong generators. The result lives in `NormallyOrderedExpressions` (products there are nested to the right, as in `01_getting_started.ipynb`), and `S(e)` evaluates it again. When the monomials satisfy relations, `express` chooses one expression; it is not canonical.

In [5]:
x = T.no(T) + 2*T.derivative(2)
e = S.express(x)
x, e, S(e) == x

(1/4*:J J J J: + 2*:dJ dJ: + 5/2*:d^2J J:, :T T: + 2*d^2T, True)

**Elements, lift and retract.** Elements of $S$ are elements of $H$ that lie in $S$, and they print in the generators of $H$. `S.lift(s)` is $s$ as an element of $H$, and `S.retract(v)` (or `S(v)`) is $v \in H$ as an element of $S$. Elements of $S$ coerce into $H$, so sums with elements of $H$ land in $H$. The coercion is the embedding $S \to H$:

In [6]:
Ts = S.gen('T')
Ts.parent() is S, S.lift(Ts).parent() is H, S.retract(J.no(J)), (Ts + J).parent() is H

(True, True, :J J:, True)

In [7]:
H.coerce_map_from(S)

Vertex algebra morphism:
  From: Vertex subalgebra of Universal vertex algebra on generators ['J'] over Rational Field strongly generated by ['T']
  To:   Universal vertex algebra on generators ['J'] over Rational Field
  Defn: T |--> 1/2*:J J:

In [8]:
S.retract(J)

ValueError: J is not an element of Vertex subalgebra of Universal vertex algebra on generators ['J'] over Rational Field strongly generated by ['T']

`S.print_options(generators='strong')` prints elements of $S$ in its strong generators instead. Here is a basis of the weight-4 space. The option belongs to $S$, so set it back afterwards.

In [9]:
S.print_options(generators='strong'); print(S.weight_space(4).basis())
S.print_options(generators='ambient'); print(S.weight_space(4).basis())

[d^2T, :T T:]
[:dJ dJ: + :d^2J J:, 1/4*:J J J J: + 1/2*:d^2J J:]


**More than one generator.** The subalgebra generated by $T$ and ${:}JJJ{:}$ needs a second strong generator: the closure finds $T_{(3)}{:}JJJ{:} = 3J$ and adds it as `S0`. So this subalgebra is all of $H$.

In [10]:
S2 = H.subalgebra([T, J.no(J.no(J))], names=['T', 'W'])
S2.strong_generator_info(), S2.gen('S0')

([('T', 2, 0, 'given'), ('W', 3, 0, 'given'), ('S0', 1, 0, 'T_(3) W')], 3*J)

The subalgebra generated by $T$ and $W = {:}JJJJ{:}$ closes with no new generator, so $T$ and $W$ strongly generate it. Both are fixed by the automorphism $J \mapsto -J$, so $S_3$ lies in the fixed-point subalgebra $H^+$. The character of $H^+$ is $\frac12\bigl(\prod_n (1-q^n)^{-1} + \prod_n (1+q^n)^{-1}\bigr)$. The dimensions agree up to weight 15, so $S_3 = H^+$ in these weights.

In [11]:
S3 = H.subalgebra([T, J.no(J.no(J.no(J)))], names=['T', 'W'])
dims = [S3.weight_space(n).dimension() for n in range(16)]
PS.<q> = PowerSeriesRing(QQ, default_prec=16)
fixed = (1/prod(1 - q^n for n in range(1, 16)) + 1/prod(1 + q^n for n in range(1, 16)))/2
S3.variable_names(), dims, dims == fixed.padded_list(16)

(('T', 'W'), [1, 0, 1, 1, 3, 3, 6, 7, 12, 14, 22, 27, 40, 49, 69, 86], True)

## 2. Exact answers and answers up to a weight

The library says in each name and result how far an answer goes.

* **Subalgebras are exact.** A subalgebra exists only after its closure has finished, so its strong generators are complete, and its weight spaces and membership tests are exact in every weight. If the closure does not finish within `max_closure_steps` rounds (default 10), the constructor raises `StrongGenerationError` and gives no partial answer. With one round, the closure for $T$ and ${:}JJJ{:}$ cannot confirm that `S0` adds nothing:

In [12]:
H.subalgebra([T, J.no(J.no(J))], names=['T', 'W3'], max_closure_steps=1)

StrongGenerationError: no strong generating set found within 1 closure round(s) (the subalgebra may not be finitely strongly generated); strong generators so far: T (weight 2), W3 (weight 3), S0 (weight 1); 3 pair(s) of generators untested; use a larger max_closure_steps

* **Ideals and quotients are exact weight by weight** (sections 3 and 4). Each weight space is computed exactly, but a list of dimensions or a character `Q.character(N)` says nothing about weights $\ge N$.
* **Some constructions are known only up to a weight.** Their methods carry the limit in the name. A commutant does not come with a complete set of strong generators. `C.strong_generators_up_to(W)` returns elements whose normally ordered monomials span every weight space of $C$ up to weight $W$; it claims nothing above $W$. For two free bosons $a, b$, the commutant of $a + b$ contains $b - a$, and up to weight 4 that one field strongly generates it (commutants are the topic of `07_commutants_and_cosets.ipynb`):

In [13]:
H2 = UniversalVertexAlgebra(QQ, ['a', 'b'], {('a', 'a'): [0, 1], ('b', 'b'): [0, 1]},
                            degrees={'a': 1, 'b': 1})
a, b = H2.gens()
C = H2.commutant([a + b])
C.strong_generators_up_to(4), [C.weight_space(n).dimension() for n in range(6)]

([-a + b], [1, 1, 2, 3, 5, 7])

* Zhu algebras have a method `is_known_exact()`. It returns `True` when the computed algebra is known to be the Zhu algebra $A(V)$, and `False` when it is only known to map onto $A(V)$ (see `09_zhu_algebras.ipynb`).

## 3. Ideals

`V.ideal(gens)` is the ideal generated by `gens`: the smallest subspace containing them that is stable under $\partial$ and under all products $a_{(n)}$, $a \in V$, $n \in \mathbb Z$. An ideal is a graded subspace of $V$; it is not a vertex algebra. Each weight space of the ideal is computed exactly, from the modes of the strong generators of $V$ applied to the generators of the ideal [KK2023].

The example is the universal affine vertex algebra of $\mathfrak{sl}_2$ at level 1, with generators $E, H, F$ and the Sugawara conformal vector, and the ideal generated by ${:}EE{:}$.

In [14]:
A = va.Affine(QQ, ['A', 1], 1, names=['E', 'H', 'F'], conformal_vector='sugawara')
E, Hc, F = A.gens()
I = A.ideal([E.no(E)], names=['EE'])
I, [A.weight_space(n).dimension() for n in range(7)], [I.weight_space(n).dimension() for n in range(7)]

(Ideal of Universal affine vertex algebra of type ['A', 1] at level 1 over Rational Field generated by [:E E:],
 [1, 3, 9, 22, 51, 108, 221],
 [0, 0, 5, 15, 38, 89, 192])

The weight-2 space of the ideal is 5-dimensional; ${:}EE{:}$ has $\mathfrak{sl}_2$-weight 4, so the zero modes generate a 5-dimensional representation from it. Its basis is in reduced echelon form. The pivots, the *leading monomials* of the ideal, are the basis monomials of $A$ that the ideal eliminates; `leading_monomials()` returns them as basis keys.

In [15]:
B = I.weight_space(2)
B.basis()

[:F F:, :H F: + dF, -2*:E F: + :H H: + dH, :E H: + dE, :E E:]

**Membership and certificates.** `I.contains(v)`, or `v in I`, is exact. `I.express(v)` writes an element of the ideal as an explicit combination of modes applied to the generator (named `EE` above). This is a certificate that $v$ lies in the ideal. Here ${:}FF{:} = \frac1{24}F_{(0)}^4\,{:}EE{:}$.

In [16]:
F.no(F) in I, E.no(Hc) + E.derivative() in I, E.no(F) in I

(True, True, False)

In [17]:
c1 = I.express(F.no(F))
c1, c1.element() == F.no(F)

(1/24*F_(0) F_(0) F_(0) F_(0) EE, True)

**Normal forms.** `I.normal_form(v)` subtracts from $v$ an element of the ideal so that no leading monomial of the ideal remains. Two elements have the same normal form exactly when their difference lies in the ideal, and the normal form is zero exactly when $v \in I$. The normal form of ${:}EH{:}$ is $-\partial E$, because ${:}EH{:} + \partial E \in I$.

In [18]:
I.normal_form(E.no(Hc)), I.normal_form(E.no(F)), I.normal_form(F.no(F))

(-dE, :E F:, 0)

## 4. Quotients

`V.quotient(I)`, or `V / I`, is the quotient vertex algebra. `I` can be an ideal or a list of its generators. An element of $V/I$ is stored as its normal form, so it prints as a combination of the *standard monomials*, the basis monomials of $V$ that are not leading monomials of $I$. `Q(v)` is the image of $v \in V$. Elements of $V$ coerce into $Q$, and `Q.lift(x)` returns the canonical representative of $x$ in $V$. Products are computed in $V$ and reduced to normal form.

In [19]:
Q = A / I
Q is A.quotient(I), Q.conformal_vector()

(True, None)

As everywhere, no conformal vector is chosen by default. `conformal_vector='ambient'` takes the image of the Sugawara vector:

In [20]:
Q = A.quotient(I, conformal_vector='ambient')
Q.central_charge()

1

In [21]:
Eq, Hq, Fq = Q.gens()
Eq.no(Eq), Eq.no(Hq), Fq.no(Fq), Eq.ope(Fq)

(0, -dE, 0, [H, 1])

In [22]:
x = Q(E.no(Hc))
x, x.parent() is Q, Q.lift(x), Q.lift(x).parent() is A

(-dE, True, -dE, True)

The weight spaces of $Q$ have the standard monomials as a basis:

In [23]:
Q.weight_space(2).basis()

[:E F:, dE, dH, dF]

The character of $Q$ agrees up to $q^9$ with that of the lattice vertex algebra $V_{A_1}$ of the root lattice $\mathbb Z\alpha$, $\alpha^2 = 2$, with conformal vector $\frac14{:}JJ{:}$. Its character is $\sum_{n \in \mathbb Z} q^{n^2} / \prod_{m \ge 1}(1-q^m)$ (lattice vertex algebras are in `05_lattice_vertex_algebras.ipynb`). This compares dimensions only, up to weight 9.

In [24]:
LA = va.Lattice(QQ, matrix([[2]]))
LA = LA.with_conformal_vector(LA.J(0).no(LA.J(0))/4)
Q.character(10), Q.character(10) == LA.character(10)

(1 + 3*q + 4*q^2 + 7*q^3 + 13*q^4 + 19*q^5 + 29*q^6 + 43*q^7 + 62*q^8 + 90*q^9 + O(q^10),
 True)

## 5. Minimal models

For coprime $p, p' \ge 2$ let $c_{p,p'} = 1 - 6(p-p')^2/(pp')$. The minimal model vertex algebra of central charge $c_{p,p'}$ [BPZ1984] is the simple quotient $L(c_{p,p'}, 0)$ of the universal Virasoro vertex algebra $V^c$. Its character was found by Rocha-Caridi [RC1985]:

$$\chi_{p,p'}(q) = \frac{\sum_{k \in \mathbb Z} \bigl(q^{k(kpp' + p - p')} - q^{(kp+1)(kp'+1)}\bigr)}{\prod_{n \ge 1}(1-q^n)}.$$

This section finds the singular vectors of $V^c$ for a parameter $c$, builds the quotients, and compares them with $\chi_{p,p'}$.

**Where singular vectors exist.** A singular vector of weight $N$ is a $v \in V^c_N$ with $L_n v = 0$ for all $n \ge 1$. Since $L_1$ and $L_2$ generate all $L_n$, $n \ge 1$, it suffices that $L_1 v = L_2 v = 0$. The function below writes the map $v \mapsto (L_1 v, L_2 v)$ on $V^c_N$ as a matrix over $\mathbb Q[c]$ (the basis of $V^c_N$ does not depend on $c$). Its columns are independent for generic $c$, and they become dependent exactly at the roots of the gcd $D_N(c)$ of its maximal minors. So the roots of $D_N$ are exactly the central charges with a singular vector of weight $N$.

In [25]:
R.<c> = QQ['c']
Vc = va.Virasoro(R, c, conformal_vector='L')

def singular_matrix(V, N):
    # the matrix of v |-> (L_1 v, L_2 v) on the weight space V_N
    L = V.gen('L')
    basis = V.weight_space(N).basis()
    rows = []
    for n in (1, 2):
        images = [L.mode(v, n) for v in basis]
        rows += [[im.coefficient(k) for im in images] for k in V.weight_space(N - n).basis_keys()]
    return matrix(V.base_ring(), rows)

D = {N: gcd(singular_matrix(Vc, N).minors(Vc.weight_space(N).dimension())) for N in (4, 6, 8)}
{N: d.factor() for N, d in D.items()}

{4: c + 22/5, 6: (c - 1/2) * (c + 68/7), 8: (c + 3/5) * (c + 46/3)}

Each root is a minimal-model central charge $c_{p,p'}$ with $(p-1)(p'-1) = N$: $-22/5 = c_{2,5}$ at $N = 4$ (Lee–Yang), $1/2 = c_{3,4}$ (Ising) and $-68/7 = c_{2,7}$ at $N = 6$, $-3/5 = c_{3,5}$ and $-46/3 = c_{2,9}$ at $N = 8$. The cell below finds $(p, p')$ for each root. At $c = c_{p,p'}$, `V.find_singular(N)` returns a basis of the singular vectors of weight $N$ (killed by the positive modes of the strong generators). For Lee–Yang it is $\Lambda = {:}LL{:} - \frac3{10}\partial^2 L$, for Ising a vector of weight 6:

In [26]:
def minimal_pair(c0, N):
    return next((p, pp) for p in range(2, N + 2) for pp in range(p + 1, N + 2)
                if gcd(p, pp) == 1 and (p - 1)*(pp - 1) == N and 1 - 6*(p - pp)^2/(p*pp) == c0)

cases = [(N, c0, minimal_pair(c0, N)) for N, d in D.items() for c0, _ in d.roots()]
LeeYang = va.Virasoro(QQ, -22/5, conformal_vector='L')
Ising = va.Virasoro(QQ, 1/2, conformal_vector='L')
cases, LeeYang.find_singular(4), Ising.find_singular(6)

([(4, -22/5, (2, 5)),
  (6, 1/2, (3, 4)),
  (6, -68/7, (2, 7)),
  (8, -3/5, (3, 5)),
  (8, -46/3, (2, 9))],
 [:L L: - 3/10*d^2L],
 [:L L L: + 93/64*:dL dL: - 33/16*:d^2L L: - 9/128*d^4L])

**The quotients.** For each case the cell builds $Q = V^{c_{p,p'}}/\langle v\rangle$, with $v$ the singular vector of weight $N$, and compares its dimensions with $\chi_{p,p'}$ up to weight 20. In the Lee–Yang quotient ${:}LL{:} = \frac3{10}\partial^2L$.

In [27]:
def vacuum_character(p, pp, prec):
    # Rocha-Caridi's vacuum character of M(p, p'), up to q^(prec-1)
    PS.<q> = PowerSeriesRing(ZZ, default_prec=prec)
    numer = PS(0)
    for k in range(-prec, prec + 1):
        for e, sign in ((k*(k*p*pp + p - pp), 1), ((k*p + 1)*(k*pp + 1), -1)):
            if 0 <= e < prec:
                numer += sign*q^e
    return numer / prod(1 - q^n for n in range(1, prec))

results = []
for N, c0, (p, pp) in cases:
    V0 = va.Virasoro(QQ, c0, conformal_vector='L')
    Q0 = V0.quotient(V0.find_singular(N), conformal_vector='ambient')
    dims = [Q0.weight_space(n).dimension() for n in range(21)]
    results.append(((p, pp), c0, dims == vacuum_character(p, pp, 21).padded_list(21)))
    if (p, pp) == (2, 5):
        LY = Q0
Lq = LY.gen('L')
results, Lq.no(Lq), LY.character(12)

([((2, 5), -22/5, True),
  ((3, 4), 1/2, True),
  ((2, 7), -68/7, True),
  ((3, 5), -3/5, True),
  ((2, 9), -46/3, True)],
 3/10*d^2L,
 1 + q^2 + q^3 + q^4 + q^5 + 2*q^6 + 2*q^7 + 3*q^8 + 3*q^9 + 4*q^10 + 4*q^11 + O(q^12))

In all five cases the quotient by the singular vector of weight $(p-1)(p'-1)$ has the dimensions of $L(c_{p,p'}, 0)$ up to weight 20. The computation shows no more than that. That the quotient is simple in every weight is Wang's theorem [Wan1993]: at $c = c_{p,p'}$ this singular vector generates the maximal ideal of $V^c$.

## 6. Presentations

A subalgebra $S$ with strong generators $s_1, \dots, s_r$ is a quotient $F/R$. Here $F$ is the universal vertex algebra on generators $s_1, \dots, s_r$ [DSK2005] whose OPEs are those of $S$, written in normally ordered monomials of the $s_i$. $R$ is the kernel of the map $F \to S$ that sends each normally ordered monomial to the same monomial in $S$. This map respects all products, so $R$ is an ideal and $F/R \cong S$. The OPEs need not satisfy the Jacobi identity in $F$ (then $F$ is only a *proto* vertex algebra, and the failures lie in $R$); `F.jacobi_identity_failures()` lists them.

`S.presentation()` returns $F/R$. `P.ambient()` is $F$, and `P.relations(w)` is a basis of $R_w$, computed exactly as the kernel in weight $w$. Elements convert both ways: `P(s)` for $s \in S$, `S(x)` for $x \in P$.

The Virasoro field $T = \frac12{:}\partial\psi\,\psi{:}$ of a Majorana fermion $\psi(z)\psi(w) \sim 1/(z-w)$ generates a Virasoro algebra at $c = 1/2$. The presentation has $F = V^{1/2}$, no relation below weight 6, and at weight 6 the Ising singular vector of section 5:

In [28]:
Fer = va.FreeFermions(QQ)
psi = Fer.gen(0)
SI = Fer.subalgebra([psi.derivative().no(psi)/2], names=['L'])
P = SI.presentation()
FL = P.ambient()
FL, FL.gen('L').ope(FL.gen('L')), [len(P.relations(w)) for w in range(7)]

(Universal vertex algebra on generators ['L'] over Rational Field,
 [dL, 2*L, 0, 1/4],
 [0, 0, 0, 0, 0, 0, 1])

In [29]:
r6 = P.relations(6)[0]
r6, repr(r6) == repr(Ising.find_singular(6)[0])

(:L L L: + 93/64*:dL dL: - 33/16*:d^2L L: - 9/128*d^4L, True)

The dimensions of $S$ agree with Rocha-Caridi's character of $L(1/2, 0)$ up to weight 20. Section 5 showed the same for the quotient of $V^{1/2}$ by the ideal generated by this singular vector. That ideal lies in $R$, so up to weight 20 it is all of $R$, and $S$ agrees in these weights with the Ising quotient of section 5.

In [30]:
[SI.weight_space(w).dimension() for w in range(21)] == vacuum_character(3, 4, 21).padded_list(21)

True

Conversions between $S$ and $P$: the normally ordered cube of $L$ is not a standard monomial of $P$, so it prints as its normal form.

In [31]:
L = P.gen('L')
x = L.no(L.no(L))
Ls = SI.gen('L')
x, SI(x) == Ls.no(Ls.no(Ls)), P(Ls) == L

(-93/64*:dL dL: + 33/16*:d^2L L: + 9/128*d^4L, True, True)

For the subalgebra $S_3$ of section 1, strongly generated by $T$ and $W = {:}JJJJ{:}$ of weights 2 and 4, the OPEs satisfy the Jacobi identity in $F$, and the first relation is at weight 8. $F$ and $S_3$ have the same dimensions below weight 8.

In [32]:
P3 = S3.presentation()
F3 = P3.ambient()
(F3.jacobi_identity_failures(), [F3.weight_space(n).dimension() for n in range(12)],
 [S3.weight_space(n).dimension() for n in range(12)], [len(P3.relations(w)) for w in range(12)])

({},
 [1, 0, 1, 1, 3, 3, 6, 7, 13, 16, 26, 33],
 [1, 0, 1, 1, 3, 3, 6, 7, 12, 14, 22, 27],
 [0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 4, 6])

## Not there yet

* Subalgebras need homogeneous generators of positive conformal weight (or a grading that makes some twisted weight positive, `horizontal_grading='ambient'`). If the closure does not finish, there is no partial result.
* Ideals and quotients need the conformal weights of $V$ and a finite set of strong generators of $V$. Their weight spaces are computed one at a time, so a statement about all weights (simplicity, a character in closed form) is a theorem to cite, not an output.
* Singular vectors are found for a fixed central charge (`find_singular`); a search over a parameter, as in section 5, is done by hand.

## References

- [BPZ1984] A. A. Belavin, A. M. Polyakov, A. B. Zamolodchikov, *Infinite conformal symmetry in two-dimensional quantum field theory*, Nuclear Phys. B 241 (1984), 333–380, [doi:10.1016/0550-3213(84)90052-X](https://doi.org/10.1016/0550-3213(84)90052-X) — the minimal models (section 5).
- [DSK2005] A. De Sole, V. G. Kac, *Freely generated vertex algebras and non-linear Lie conformal algebras*, Comm. Math. Phys. 254 (2005), 659–694, [arXiv:math-ph/0312042](https://arxiv.org/abs/math-ph/0312042) — universal vertex algebras (section 6).
- [KK2023] R. A. Kozlov, P. S. Kolesnikov, *The Gröbner–Shirshov bases method for vertex algebras*, J. Math. Sci. 296 (2026), 217–230, [arXiv:2312.01670](https://arxiv.org/abs/2312.01670) — ideals and normal forms (sections 3, 4).
- [RC1985] A. Rocha-Caridi, *Vacuum vector representations of the Virasoro algebra*, in: Vertex operators in mathematics and physics, Math. Sci. Res. Inst. Publ. 3, Springer (1985), 451–473, [doi:10.1007/978-1-4613-9550-8_22](https://doi.org/10.1007/978-1-4613-9550-8_22) — the vacuum characters of the minimal models (sections 5, 6).
- [Wan1993] W. Wang, *Rationality of Virasoro vertex operator algebras*, Int. Math. Res. Not. 1993, no. 7, 197–211, [doi:10.1155/S1073792893000212](https://doi.org/10.1155/S1073792893000212) — the singular vector generates the maximal ideal (section 5).